# Comparator — figures

Loads `results/comparator_data_2digit.npz` from `data_comparator.py`. No model, no GPU: every cell
here is plotting.

Base patch for everything: a rank-2 interchange of the **pre-MLP L14 residual inside
span($\mathbf{v}_1,\mathbf{v}_2$)** — the summed plane the comparator reads, rather than the two
rank-1 sources.

Sections: conditions · module freezes · attribution and top-$k$ · overlap between cases ·
injection · the plane · receptive fields · geometry · boundaries · connectivity · DAS.

In [ ]:
import os, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)
TAG = '2digit'

Z = dict(np.load(f'{RESULTS_DIR}/comparator_data_{TAG}.npz'))
M = json.load(open(f'{RESULTS_DIR}/comparator_data_meta.json'))
CASES = M['cases']
ck = (lambda s: s.replace(' ', '_'))
NL = M['neuron_layers']

print(f"{TAG}: base = {M['base_patch']}")
print(f"cases {CASES}   n_eval {M['n_eval']}   pool {M['d_ff'] * len(NL)} neurons   "
      f"runtime {M['runtime_s']:.0f} s")
print(f"{len(Z)} arrays loaded")

## Plot style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 16, 13, 13, 17
DPI, SAVE = 300, True

COL = {'full': '#2166AC', 'das': '#C97B00', 'pc1': '#C06A9B', 'h14': '#117777', 'ceil': '#6E6E6E'}
CASE_COL = {CASES[0]: COL['das'], CASES[1]: COL['full']}
V1, V2 = r'$\mathbf{v}_1$', r'$\mathbf{v}_2$'


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for s in ('top', 'right'):
        if s in ax.spines: ax.spines[s].set_visible(False)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/comparator_{TAG}_{name}.png', dpi=DPI, bbox_inches='tight')


def mean_se(key):
    v = Z[key]
    return v.mean(), v.std() / np.sqrt(len(v))


# Ranking, rebuilt from the saved attribution.
POOL_LAYER, POOL_WITHIN = Z['pool_layer'], Z['pool_within']
ORDER = {c: Z[f'order_{ck(c)}'] for c in CASES}          # top 1000 per case
SCORES = {c: np.concatenate([Z[f'attr_{ck(c)}_L{l}'] for l in NL]) for c in CASES}
PROF, PL, PW = Z['prof_pooled'], Z['prof_layer'], Z['prof_within']
NAME = [f'L{l}n{w}' for l, w in zip(PL, PW)]
RANK = {c: Z[f'prof_rank_{ck(c)}'] for c in CASES}
print('profiled neurons:', len(PROF), f'(L14 {(PL == 14).sum()}, L15 {(PL == 15).sum()})')

### The direction that flips the answer

Normal to the dose-response contours of `makefig_sharedreps` panel (f), which live in this same
span($\mathbf{v}_1,\mathbf{v}_2$) plane. Logistic fit of the sweep outcome on the two plane
coordinates; the weight vector is the contour normal. Reference arrow in panel (a) below.


In [ ]:
# The dose-response contours of makefig_sharedreps' panel (f) live in this same span(v1, v2)
# plane: that sweep SETS the two coordinates to what a chosen pair of numbers would put there and
# records which slot the model then names. The direction orthogonal to those contours is the
# direction that flips the answer, so it is the causal reference the read arrows in the summary
# figure's (a) are worth comparing against.
#
# Easier than contouring and then differentiating: a logistic fit of the per-prompt outcome on the
# two coordinates has straight, parallel iso-probability contours, so its weight vector IS the
# contour normal. Cross-checked against the drawn 0.5 contour below.
SW  = dict(np.load(f'{RESULTS_DIR}/sharedrep_{TAG}.npz'))
SWM = json.load(open(f'{RESULTS_DIR}/sharedrep_meta.json'))['regimes'][TAG]['meta']

# make_batch names tok_nc after the WINNING slot, which is the first number only in 'n1 larger';
# reading "did it name the first slot" straight off tok_nc inverts the other case.
_nc, _b = SW['sweep_tok_prompt_nc'], SW['sweep_tok_prompt']
T1, T2 = (_nc, _b) if SWM['case'] == 'n1 larger' else (_b, _nc)
_sam, _sc = SW['sweep_argmax'], SW['sweep_coords'].astype(float)
G = len(_sc)

# One row per prompt that named one of the two implanted numbers, with the plane coordinates that
# cell was set to. A constant offset cannot move a gradient, so the sweep's own centring never has
# to be reconciled with the cloud's.
_X, _y = [], []
for i in range(G):
    for j in range(G):
        if i == j:                                   # the sweep skips y1* == y2*
            continue
        for k, a in enumerate(_sam[i, j]):
            if a == T1[k] or a == T2[k]:
                _X.append((_sc[i, 0], _sc[j, 1])); _y.append(float(a == T1[k]))
_X, _y = np.array(_X, float), np.array(_y)


def logit_fit(X, y, iters=100):
    """Newton / IRLS logistic fit. Returns the weight vector: the normal to the iso-probability
    contours, pointing toward y = 1."""
    A = np.c_[X - X.mean(0), np.ones(len(X))]
    w = np.zeros(A.shape[1])
    for _ in range(iters):
        p = np.clip(1 / (1 + np.exp(-A @ w)), 1e-9, 1 - 1e-9)
        H = A.T @ (A * (p * (1 - p))[:, None]) + 1e-8 * np.eye(A.shape[1])
        w = w + np.linalg.solve(H, A.T @ (y - p))
    return w[:-1]


_w = logit_fit(_X, _y)
BND_NORMAL = _w / np.linalg.norm(_w)             # points toward "the model answers y1"

# ---- cross-check: total-least-squares normal of the 0.5 contour the sharedreps panel draws -----
_ang = lambda v: np.degrees(np.arctan2(v[1], v[0]))
_P = np.full((G, G), np.nan)
for i in range(G):
    for j in range(G):
        if i != j:
            _n1, _n2 = (_sam[i, j] == T1).sum(), (_sam[i, j] == T2).sum()
            _P[i, j] = _n1 / max(_n1 + _n2, 1)
for k in range(G):                    # contouring cannot cross the missing diagonal
    _P[k, k] = np.mean([_P[k, q] for q in (k - 1, k + 1) if 0 <= q < G])
_f, _a = plt.subplots()
_pts = np.vstack([p.vertices for p in _a.contour(_sc[:, 0], _sc[:, 1], _P.T,
                                                 levels=[0.5]).get_paths()])
plt.close(_f)
_n5 = np.linalg.svd(_pts - _pts.mean(0), full_matrices=False)[2][1]
_n5 *= np.sign(_n5 @ BND_NORMAL)
print(f'boundary normal {_ang(BND_NORMAL):+6.1f} deg  '
      f'(0.5-contour check {_ang(_n5):+.1f} deg, cos {float(BND_NORMAL @ _n5):.4f}; '
      f'{len(_y)} outcomes)')
print(f'difference axis {_ang(Z["diff_axis"]):+6.1f} deg  '
      f'cos with the boundary normal {float(BND_NORMAL @ Z["diff_axis"]):+.3f}  -- the '
      'observational axis is not the causal one')


## Summary figure

Chosen by **attribution rank within the overlap set** (the neurons that matter in both cases), not
by how comparator-like the field looks — a neuron that compares only inside a band of values can
have a middling AUC and still be load-bearing.

- MLP14: the four highest-ranked shared neurons.
- MLP15: the highest-ranked shared neuron, which is also the cleanest comparator.

Panels: read directions in the plane · the four MLP14 receptive fields · the MLP15 receptive
field · the rank-1 L15 DAS component, coloured by which slot holds the max · IIA for the plane
patch, for that patch with the shared neurons frozen, and for the rank-1 L15 DAS patch.

In [ ]:
# ==================================================================================
# FONT SIZES  --  edit any of these
# ==================================================================================
FS_AXIS       = 28   # x / y axis labels, every panel
FS_TICK       = 22   # numeric tick labels, every panel
FS_PANEL      = 34   # the bold (a) ... (e)
FS_NEURON     = 20   # (a), (b), (c)  the neuron name inside each panel
FS_CBAR_TICK  = 21   # colorbar tick labels
FS_CBAR_LABEL = 26   # colorbar labels
FS_LEGEND     = 22   # (d), (e)  legend entries
FS_ANNOT      = 22   # (a)  labels on the arrows, the y1 = y2 line and the two groups
ANNOT_OFF     = 1.13 # (a)  right-pointing arrow labels sit at this multiple of the arrow length
HEADROOM_A    = 0.14 # (a)  extra space above the cloud, fraction of the y range: y2 > y1 sits in it
FOOTROOM_A    = 0.10 # (a)  and below it: y1 > y2 sits in it, bottom right
HL_LW         = 3.2  # (b)  frame width on the receptive fields drawn as arrows in (a)
COMPASS_BOX_A = [0.0, 0.0, 0.22, 0.27]   # (a)  v1 / v2 compass, axes coords: the empty lower left
FS_BAR_TICK   = 22   # (e)  labels under the bars

# ==================================================================================
# LAYOUT  --  edit any of these
# ==================================================================================
FIG_SIZE     = (23.5, 14.6)
ROW_HEIGHTS  = (1.38, 1.0)   # rows near-equal: (b)'s squares match (c) and (a) fills its row
GRID_WSPACE, GRID_HSPACE = 0.40, 0.34      # between the outer panels
MARGINS = dict(left=0.055, right=0.925, top=0.94, bottom=0.075)   # room for the (b) colorbar
CBAR_W, CBAR_GAP = 0.011, 0.014            # colorbar width and its gap, figure fractions
RF_WSPACE, RF_HSPACE = 0.04, 0.14          # between the six receptive fields in (b)
PANEL_LX, PANEL_LY = -0.16, 1.03
RF_TICKS = [20, 50, 80]                    # sparse ticks so the small panels stay readable
# Receptive fields: trimmed PRGn (green positive, purple negative), the same map as
# makefig_multicompare (c). Not red/blue: blue already means "y2" in (d) and (e).

FIN_L15 = 12784          # the MLP15 neuron drawn in (a) and (c); n8383 is the alternative
FIN_OVK = 20             # the overlap depth whose frozen set is bar 2 of (e)
FIN_DASRANK = 1
FIN_DASFIT = CASES[0]    # (d), (e) the ONE L15 DAS direction, fitted with y1 perturbed; (e) evaluates
                         #     this same direction in both cases instead of refitting per case
ARROW_N14 = [6150, 9459]   # (a) the MLP14 neurons whose gate read directions are drawn
ARROW_LEN = 0.62         # (a) every arrow this fraction of the axes box, so the
                         #     cloud sets the scale; the in-plane fraction goes in the legend
PLANE_BOX_ASPECT = 0.952 # (a) height / width of the axes box, = makefig_sharedreps panel (a)
DIFF_LINE_EXT = 0.62     # (a) length of the y1 = y2 marker, as a fraction of the axes box -- a
                         #     reference mark through the cloud, not a divider across the panel
CLOUD_S, CLOUD_ALPHA = 15, 0.50    # (a) the cloud the arrows sit on
# Coloured by which number is larger, in makefig_sharedreps' own convention: warm = the first
# number, cool = the second (its y1 / y2 ramps, and the PuOr_r dose-response surface). These are
# the two colours panel (d) already splits its histogram with, so one slot is one colour
# throughout the figure. Kept pale: the read arrows have to stay legible over them.
SLOT_COL = {True: COL['das'], False: COL['full']}      # True = y1 > y2

# The case names say which slot holds the clean maximum; what a reader needs is which number the
# counterfactual perturbs, which is the same fact stated directly.
CASE_LAB = {CASES[0]: r'$y_1$ perturbed', CASES[1]: r'$y_2$ perturbed'}

import matplotlib.patheffects as pe
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap


def trim_cmap(name, t, n=256):
    """`name` with a fraction `t` removed from each end (makefig_sharedreps' truncate_cmap,
    applied to the dark ends of a diverging map rather than the pale ends of a sequential one)."""
    return LinearSegmentedColormap.from_list(
        f'{name}_t{t}', matplotlib.colormaps[name](np.linspace(t, 1 - t, n)))


RF_CM = trim_cmap('PRGn', 0.16)   # drop 16% off each end: PRGn's extremes are very dark

# ---- selection: the MLP14 half of the 12 shared comparator neurons -----------------------------
# The same set (e)'s bar 2 freezes: the neurons in BOTH cases' top-FIN_OVK attribution lists.
# Ordered by worst-case rank, as in the appendix's receptive-field figure. Not by how a field looks.
ov = set(int(x) for x in Z[f'overlap_k{FIN_OVK}'])
cand = sorted([(max(RANK[CASES[0]][j], RANK[CASES[1]][j]), int(PL[j]), int(PW[j]), j)
               for j in range(len(PL)) if int(PROF[j]) in ov])
rows14 = [c[3] for c in cand if c[1] == 14]
assert len(ov) == 12 and len(rows14) == 6, (len(ov), len(rows14))
row15 = next(j for j in range(len(PL)) if PL[j] == 15 and PW[j] == FIN_L15)
col_of = lambda r: int((PL[:r] == PL[r]).sum())
read_of = lambda r: Z[f'read_gate_prof_L{int(PL[r])}'][col_of(r)].astype(float)
frac_of = lambda r: float(Z[f'readfrac_gate_prof_L{int(PL[r])}'][col_of(r)])
rf_of = lambda r: Z[f'rf_acts_L{int(PL[r])}'][:, col_of(r)]

chance = np.sqrt(2 / M['d_model'])
ang_of = lambda r: np.degrees(np.arctan2(*read_of(r)[::-1]))
arrows = [next(j for j in range(len(PL)) if PL[j] == 14 and PW[j] == w) for w in ARROW_N14]

grid = Z['rf_grid']
ext = [grid.min(), grid.max(), grid.min(), grid.max()]
pq, ax_diff = Z['plane_pq'], Z['diff_axis']
y1c, y2c = Z['cloud_y1'].astype(float), Z['cloud_y2'].astype(float)
# The cloud owns amber and blue now, so the read arrows take the rest of the palette.
# Arrow colours, in ARROW_N14 order. Crimson for the second: clear over both clouds and CVD-distinct
# (min OKLab dE >= 16 under normal, deutan and protan vision) from blue, amber, black and PRGn.
ACOL = ['#111111', '#B2182B', COL['pc1'], '#7B3294', COL['ceil']]

fig = plt.figure(figsize=FIG_SIZE)
gs = fig.add_gridspec(2, 3, wspace=GRID_WSPACE, hspace=GRID_HSPACE, height_ratios=ROW_HEIGHTS)


def panel_letter(ax, s):
    ax.text(PANEL_LX, PANEL_LY, s, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight='bold', ha='left', va='bottom')


def rf_axes(ax, r, show_x, show_y, hl=None):
    """One receptive field, normalised to its own peak so the panels share a colour scale. `hl`, a
    colour: the neuron is one of (a)'s arrows -- frame and name drawn in that arrow's colour."""
    a = rf_of(r).reshape(len(grid), len(grid)).T
    a = a / np.abs(a).max()
    ax.imshow(a, origin='lower', extent=ext, aspect='equal', cmap=RF_CM, vmin=-1, vmax=1)   # square
    ax.plot(ext[:2], ext[:2], '--', color='0.25', lw=1.0)
    ax.text(0.04, 0.955, f'#{PW[r]}', transform=ax.transAxes, fontsize=FS_NEURON, va='top',
            color=hl or 'black', fontweight='bold' if hl else 'normal',
            bbox=dict(boxstyle='round,pad=0.16', fc='white', ec='none', alpha=0.8))
    if hl:
        for sp in ax.spines.values():
            sp.set_edgecolor(hl); sp.set_linewidth(HL_LW)
    ax.set_xticks(RF_TICKS); ax.set_yticks(RF_TICKS)
    ax.tick_params(labelsize=FS_TICK)
    ax.set_xlabel('$y_1$', fontsize=FS_AXIS) if show_x else ax.set_xticklabels([])
    ax.set_ylabel('$y_2$', fontsize=FS_AXIS) if show_y else ax.set_yticklabels([])


# ===================================== (a) read directions in the plane ========================
ax = fig.add_subplot(gs[0, 0])
Rad = np.quantile(np.linalg.norm(pq, axis=1), 0.97)
slot0 = y1c > y2c
for _first, _m, _lab in ((True, slot0, '$y_1 > y_2$'), (False, ~slot0, '$y_2 > y_1$')):
    ax.scatter(pq[_m, 0], pq[_m, 1], s=CLOUD_S, color=SLOT_COL[_first],
               alpha=CLOUD_ALPHA, edgecolors='none', zorder=0)
# Each group named beside its own cloud: y2 > y1 lies on the left of the plane, y1 > y2 on the right.
_halo = [pe.withStroke(linewidth=4, foreground='white')]
# Both on empty space: y2 > y1 in the headroom above its cloud, y1 > y2 in the strip below its own.
for _first, _lab, _xy, _ha in ((False, '$y_2 > y_1$', (0.10, 0.955), 'left'),
                               (True,  '$y_1 > y_2$', (0.985, 0.035), 'right')):
    ax.text(*_xy, _lab, transform=ax.transAxes, fontsize=FS_ANNOT, color=SLOT_COL[_first],
            fontweight='bold', ha=_ha, va='center', zorder=6, path_effects=_halo)
# Same box and data-fitted limits as makefig_sharedreps panel (a) (default 5% margins), so the two
# panels of the same plane have the same shape. lim_x / lim_y are the half-ranges the arrows and
# the y1 = y2 marker are scaled by.
ax.set_box_aspect(PLANE_BOX_ASPECT)
ax.set_anchor('N')          # fixed shape in a taller row: sit at the top, level with (b)
ax.autoscale_view()
_y0, _y1 = ax.get_ylim(); ax.set_ylim(_y0 - FOOTROOM_A * (_y1 - _y0), _y1 + HEADROOM_A * (_y1 - _y0))
lim_x, lim_y = np.ptp(ax.get_xlim()) / 2, np.ptp(ax.get_ylim()) / 2

# The y1 = y2 boundary: normal to the measured difference axis, drawn in data coordinates so it
# actually divides the cloud. The arrows are drawn in the same frame, so "reads along" versus
# "reads across" the boundary can be read off the panel. Because the two axes are scaled
# independently, that frame is sheared: angles here are not the angles in the (v1, v2) basis, and
# the true ones are printed under the cell.
_d = np.array([-ax_diff[1], ax_diff[0]], dtype=float)          # along the boundary
t = np.linspace(-1, 1, 2) * DIFF_LINE_EXT / np.hypot(_d[0] / lim_x, _d[1] / lim_y)
ax.plot(_d[0] * t, _d[1] * t, '--', color='0.30', lw=1.6, zorder=2)
_end = _d * t.max() if _d[1] * t.max() > 0 else _d * t.min()        # the line's upper end
ax.text(*(_end * 1.04), '$y_1 = y_2$', fontsize=FS_ANNOT, color='0.30', ha='left', va='bottom',
        zorder=6, path_effects=_halo)


def plane_arrow(u, col, lw, ms):
    """One unit direction, scaled so every arrow is the same fraction of the axes box."""
    u = np.asarray(u, float) / np.linalg.norm(u)
    v = u * ARROW_LEN / np.hypot(u[0] / lim_x, u[1] / lim_y)
    ax.annotate('', xy=v, xytext=(0, 0), zorder=3,
                arrowprops=dict(arrowstyle='-|>', lw=lw, color=col, shrinkA=0, shrinkB=0,
                                mutation_scale=ms))
    return v


# The dose-response boundary normal is deliberately NOT drawn here: at this panel's independent
# axis scaling it lands close to perpendicular to the colour split and reads as a restatement of
# it. Each arrow's true separation from it is printed under the cell instead, where no shear
# distorts it.
# Each arrow labelled at its tip with its neuron, in its own colour (all are MLP14; the caption says so).
for c_, r in enumerate(arrows):
    tip = plane_arrow(read_of(r), ACOL[c_], 2.8, 20)
    # Right-pointing: beyond the tip. Left-pointing: centred on the tip -- beyond it runs into the
    # y axis -- above it if the arrow rises, below it if it is flat (above sits under a rising one).
    if tip[0] > 0.15 * lim_x:
        xy, ha, va = tip * ANNOT_OFF, 'left', 'center'
    elif tip[1] > 0.1 * lim_y:
        xy, ha, va = tip + np.array([0.0, 0.07 * lim_y]), 'center', 'bottom'
    else:
        xy, ha, va = tip - np.array([0.0, 0.07 * lim_y]), 'center', 'top'
    ax.text(*xy, f'#{PW[r]}', fontsize=FS_ANNOT, color=ACOL[c_], fontweight='bold',
            ha=ha, va=va, zorder=6, path_effects=_halo)
ax.set_xlim(ax.get_xlim()); ax.set_ylim(ax.get_ylim())    # freeze: the arrows must not rescale it
ax.set_xticks([-5, 0, 5])                                  # as in makefig_sharedreps (a)

# v1 / v2 at their true angle, as in makefig_sharedreps: an equal-aspect inset, since the panel's
# own axes are scaled independently. Signs are the plot's own (plane_sgn), so the angle matches.
_cos = float(np.prod(Z['plane_sgn']) * (Z['v1'] @ Z['v2'])
             / (np.linalg.norm(Z['v1']) * np.linalg.norm(Z['v2'])))
axi = ax.inset_axes(COMPASS_BOX_A); axi.set_aspect('equal')
for vec, nm in (((1.0, 0.0), V1), ((_cos, np.sqrt(1 - _cos ** 2)), V2)):
    axi.annotate('', xy=vec, xytext=(0, 0),
                 arrowprops=dict(arrowstyle='-|>', lw=2.3, color='#111111', mutation_scale=15))
    axi.text(1.38 * vec[0], 1.38 * vec[1], nm, fontsize=FS_ANNOT - 2, fontweight='bold',
             ha='center', va='center')
axi.set_xlim(-0.5, 1.7); axi.set_ylim(-0.35, 1.75); axi.axis('off')
print(f'(a) compass: cos(v1, v2) = {_cos:+.3f} -> {np.degrees(np.arccos(_cos)):.1f} deg')
ax.set_xlabel(r'$f_1(y_1)$, comp. along $\mathbf{v}_1$', fontsize=FS_AXIS)
ax.set_ylabel(r'comp. along $\mathbf{v}_2^{\perp}$', fontsize=FS_AXIS)
ax.tick_params(labelsize=FS_TICK)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
panel_letter(ax, '(a)')

# ===================================== (b) the six shared MLP14 receptive fields ===============
inner = gs[0, 1:].subgridspec(2, 3, wspace=RF_WSPACE, hspace=RF_HSPACE)
bax = []
for k, r in enumerate(rows14):
    axk = fig.add_subplot(inner[k // 3, k % 3])
    rf_axes(axk, r, show_x=(k >= 3), show_y=(k % 3 == 0),
            hl=ACOL[arrows.index(r)] if r in arrows else None)
    bax.append(axk)
    if k == 0:
        panel_letter(axk, '(b)')

# ===================================== (c) the MLP15 receptive field ===========================
ax = fig.add_subplot(gs[1, 0])
rf_axes(ax, row15, show_x=True, show_y=True)
cax = ax
panel_letter(ax, '(c)')

# ===================================== (d) the comparator at the L15 residual ==================
ax = fig.add_subplot(gs[1, 1])
P = Z[f'das15_proj_{ck(FIN_DASFIT)}_k{FIN_DASRANK}'][:, 0]
P = P - P.mean()
slot0 = y1c > y2c
bins = np.linspace(P.min(), P.max(), 55)
for s_, m_, lab in ((0, slot0, '$y_1 > y_2$'), (1, ~slot0, '$y_2 > y_1$')):
    ax.hist(P[m_], bins=bins, color=[COL['das'], COL['full']][s_], alpha=0.75, label=lab)
ax.set_xlabel(f'rank-{FIN_DASRANK} DAS component, L15 residual', fontsize=FS_AXIS)
ax.set_ylabel('count', fontsize=FS_AXIS, labelpad=8)
ax.tick_params(labelsize=FS_TICK)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
ax.legend(fontsize=FS_LEGEND, frameon=False)
panel_letter(ax, '(d)')

# ===================================== (e) interventions =======================================
ax = fig.add_subplot(gs[1, 2])
n_sh = len(Z[f'overlap_k{FIN_OVK}'])
# The frozen set is, precisely, the neurons that land in BOTH cases' top-FIN_OVK attribution
# lists -- an intersection, not a rank depth. The tick label says "top-n_sh" anyway: a paper
# reader has not been shown the per-case lists, and what the bar is about is that these are the
# neurons most responsible. The exact membership is printed under the cell and set by FIN_OVK.
# Bar 3 is one direction for both cases: the FIN_DASFIT fit, evaluated in each case, which is the
# direction (d) projects onto. The per-case refits are in the appendix transfer figure.
BARS = [(f'({V1},{V2})\npatch',
         lambda c: Z[f'cond_iia_{ck(c)}_uv_plane_preMLP']),
        (f'({V1},{V2})\npatch +\nfreeze top-{n_sh}\nMLP neurons',
         lambda c: Z[f'ov{FIN_OVK}_iia_{ck(c)}_shared']),
        (f'L15 DAS\nrank {FIN_DASRANK}',
         lambda c: Z[f'das15_iia_{ck(c)}_from_{ck(FIN_DASFIT)}_k{FIN_DASRANK}'])]
xs = np.arange(len(BARS))
w = 0.8 / len(CASES)
for j, case in enumerate(CASES):
    m = [f(case).mean() for _, f in BARS]
    se = [f(case).std() / np.sqrt(M['n_eval']) for _, f in BARS]
    b = ax.bar(xs + (j - (len(CASES) - 1) / 2) * w, m, w, yerr=se, capsize=3,
               color=CASE_COL[case], alpha=0.9, edgecolor='white', linewidth=0,
               label=CASE_LAB[case])
ax.set_ylim(0, 1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for n, _ in BARS], fontsize=FS_BAR_TICK)
ax.set_ylabel('IIA', fontsize=FS_AXIS)
ax.tick_params(axis='y', labelsize=FS_TICK)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper left', ncol=2, columnspacing=1.1)
panel_letter(ax, '(e)')

# Margins first, then the colourbars into explicit axes: letting colorbar() steal space from a
# subgridspec pushes it over the last column instead of sitting beside it.
fig.subplots_adjust(**MARGINS)

# (b) and (c) share the normalisation, so only (b)'s bar is labelled -- labelling (c)'s puts
# the text straight into (d)'s y axis.
for anchors, labelled in ((bax, True), ([cax], False)):
    for a in anchors:
        a.apply_aspect()                 # square fields shrink inside their cells: measure the real box
    bb = anchors[0].get_position()
    for a in anchors[1:]:
        bb = bb.union([bb, a.get_position()])
    cb = fig.colorbar(ScalarMappable(Normalize(-1, 1), RF_CM),
                      cax=fig.add_axes([bb.x1 + CBAR_GAP, bb.y0, CBAR_W, bb.height]),
                      ticks=[-1, 0, 1])
    if labelled:
        cb.set_label('activation / peak', fontsize=FS_CBAR_LABEL)
    cb.ax.tick_params(labelsize=FS_CBAR_TICK)
savefig(fig, 'summary')
# plt.savefig('./figs_mainpaper/comparator_fig.pdf', bbox_inches='tight', dpi=300)
plt.show()

print('ranks are (y1 perturbed / y2 perturbed)')
print('(b) MLP14:  ' + '  '.join(f'#{PW[r]}({RANK[CASES[0]][r]}/{RANK[CASES[1]][r]})'
                                 for r in rows14))
print(f'(c) MLP15:  #{PW[row15]}({RANK[CASES[0]][row15]}/{RANK[CASES[1]][row15]})')
print('(a) arrows: ' + '  '.join(f'MLP{PL[r]} #{PW[r]} {ang_of(r):+.0f} deg, in-plane '
                                 f'{frac_of(r):.3f}' for r in arrows)
      + f'   (chance {chance:.3f})')
if 'BND_NORMAL' in globals():
    _bang = np.degrees(np.arctan2(BND_NORMAL[1], BND_NORMAL[0]))
    print('(a) vs the boundary normal (%+.0f deg): ' % _bang
          + '  '.join(f'MLP{PL[r]} #{PW[r]} '
                      f'{abs((ang_of(r) - _bang + 180) % 360 - 180):.0f} deg off' for r in arrows))
print(f'(e) bar 2 freezes the top-{FIN_OVK} overlap: {n_sh} neurons, '
      f'L14 {(POOL_LAYER[Z[f"overlap_k{FIN_OVK}"]] == 14).sum()} / '
      f'L15 {(POOL_LAYER[Z[f"overlap_k{FIN_OVK}"]] == 15).sum()}')
print(f'(e) bar 3: one rank-{FIN_DASRANK} L15 direction fitted with {CASE_LAB[FIN_DASFIT]}, IIA '
      + '  '.join(f'{CASE_LAB[c]} {Z[f"das15_iia_{ck(c)}_from_{ck(FIN_DASFIT)}_k{FIN_DASRANK}"].mean():.3f}'
                  for c in CASES))